### AITW-microstructures

Generates realistic wood (Birch, Spruce, etc.) microstructures from a set of given parameters using either a direct application of modeled deformation maps or a surrogate model.

#### Links

- <a href="https://github.com/AI-TranspWood/AITW_microstructures" target="_blank" rel="noopener noreferrer">Source code</a>
- <a href="https://www.eessi.io/docs/available_software/detail/AITW-microstructures" target="_blank" rel="noopener noreferrer">EESSI</a>
  
#### References

- <a href="https://doi.org/10.1016/j.nxmate.2026.102641" target="_blank" rel="noopener noreferrer">Mińkowski, Marcin, et al. Next Materials 13 (2026): 102641.</a>

In [ ]:
from utilities import initialize_with_module

await initialize_with_module(
    [
        'AITW-microstructures/1.2.0-foss-2023a',
        'matplotlib/3.7.2-gfbf-2023a',
    ],
    '2023.06'
)

In [ ]:
# Fix bug in 1.2.0
import wood_microstructure

if wood_microstructure.__version__ == '1.2.0':
    from wood_microstructure.microstructure import WoodMicrostructure
    from wood_microstructure.filter_fit_porosity import FitPorosity

    def fit_porosity(self):
        """Run the extra step to fit the porosity of the generated volume image"""
        if not self.params.fit_porosity:
            return

        final_volume = os.path.join(self.root_dir, 'FinalVolume3D', f'FinalVolume.nrrd')
        if not os.path.exists(final_volume):
            self.logger.warning('Final volume was not produced. Skipping porosity fitting.')
            return

        data = self.params.fit_porosity_params
        data['input_file'] = final_volume
        # data['output_dir'] = os.path.join(self.root_dir, 'PorosityFitting')

        output_dir = os.path.join(self.root_dir, 'PorosityFitting')

        FitPorosity.run_from_dict(
            data=data, output_dir=output_dir, loglevel=self.handler_level, rich_live=self.rich_live
        )

    # Replace the original method with the patched one
    WoodMicrostructure.fit_porosity = fit_porosity


In [ ]:
import utilities
from common import WOOD_MS_DEFAULTS
from wood_microstructure.cli.generate import generate

widget_generate, generate_event = utilities.click_cli_widget(
    generate,
    override_defaults=WOOD_MS_DEFAULTS,
    # hide_params=['fit_porosity', 'fit_porosity_config'],
    title='Generate wood microstructure',
    button_description='Generate',
    button_running_text='Generating...'
)

# display(widget_generate)

In [ ]:
# Rewrite of the above cell with autodetection of which folders contains the images
import re
import os
import json
import ipywidgets as wdg
from IPython.display import display
from IPython.display import Image as IPyImage

from common import WOOD_MS_OUTDIR

rgx_dir = re.compile(r"Save(\w+)_(\d+)")
rgx_img = re.compile(r"volImgRef_(\d+).png")

# title = wdg.HTML(value="<h2>Microstructure Output Visualization</h2>")

def find_outputs(base_path=WOOD_MS_OUTDIR):
    outputs = []
    if not os.path.exists(base_path):
        return outputs
    for path in os.listdir(base_path):
        if not os.path.isdir(os.path.join(base_path, path)):
            continue
        match = rgx_dir.match(path)
        if not match:
            continue
        material, index = match.groups()
        outputs.append((material, int(index), os.path.join(base_path, path)))

    return outputs

# Start with a choice widget to select which output to visualize
output_selector = wdg.Dropdown(
    options=[],
    description='Select Output:',
    disabled=False,
    style={'description_width': '150px'},
)
output_selector.layout.width = '500px'

def update_output_options(*args):
    """Update the options of the output selector based on the available outputs in the results folder."""
    outputs = find_outputs()
    options = []
    for material, index, path in sorted(outputs):
        params = json.load(open(os.path.join(path, "params.json")))
        surrogate = params['surrogate']
        with_ray_cell = bool(params['is_exist_ray_cell'])
        with_vessel = bool(params['is_exist_vessel'])
        options.append((
            f"{material} - {index}: surrogate={surrogate}, ray_cells={with_ray_cell}, vessels={with_vessel}",
            path
        ))
    output_selector.options = options
    if output_selector.value is None:
        output_selector.value = options[0][1] if options else None

# Observe changes in the common event to update the output options whenever a new microstructure is generated
try:
    generate_event.observe(update_output_options, names='value')
except Exception as e:
    print("common-event not defined.")
    print("Re-launch this cell after executing the previous one to enable automatic updates of the dropdown choices.")

# Select whether to show the local distortion images or the backbone images, and a slider to select which slice to show
toggle_checkbox = wdg.Checkbox(
    value=False,
    description='With local distortion',
    style={'description_width': '150px'},
)
toggle_checkbox.layout.width = '500px'

# Slider to select which slice to show, with options automatically updated based on the selected output
image_slider = wdg.SelectionSlider(
    options=[0],
    description='Slice number:',
    continuous_update=False,
    style={'description_width': '150px'},
)
image_slider.layout.width = '500px'

image_output = wdg.Output()

def update_image_options(*args):
    """Update the options of the image slider based on the selected output and toggle state."""
    selected_output = output_selector.value
    if not selected_output:
        return
    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    custom_index = []
    for file in os.listdir(os.path.join(selected_output, imgdir)):
        match = rgx_img.match(file)
        if match:
            custom_index.append(int(match.group(1)))
    custom_index.sort()

    image_slider.options = custom_index
    image_slider.value = image_slider.options[0] if image_slider.options else None
output_selector.observe(update_image_options, names='value')

def display_image(*args):
    """Display the selected image based on the selected output, toggle state, and slider value."""
    selected_output = output_selector.value
    if not selected_output:
        return

    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    index_list = image_slider.options
    image_list = [os.path.join(selected_output, imgdir, f"volImgRef_{idx:05d}.png") for idx in index_list]

    selected_custom_index = image_slider.value
    position = index_list.index(selected_custom_index)

    with image_output:
        image_output.clear_output(wait=True)
        display(IPyImage(url=image_list[position]))

# Observe changes in the toggle and slider to update the displayed image
toggle_checkbox.observe(display_image, names='value')
image_slider.observe(display_image, names='value')
output_selector.observe(display_image, names='value')

# Initial update of the output options and image options
update_output_options()

visualizer_box = wdg.VBox([output_selector, toggle_checkbox, image_slider, image_output])
visualizer_accordion = wdg.Accordion(children=[visualizer_box], selected_index=None)
visualizer_accordion.set_title(0, "Microstructure Output Visualization")
generate_box = wdg.VBox([widget_generate, visualizer_accordion])
# main_accordion = wdg.Accordion(children=[main_box], selected_index=0)
# main_accordion.set_title(0, "Generate Microstructure")
# main_vbox = wdg.VBox([output_selector, toggle_checkbox, image_slider, image_output])
# main_accordion = wdg.Accordion(children=[main_vbox], selected_index=None)
# main_accordion.set_title(0, "Microstructure Output Visualization")

# display(main_accordion)

In [ ]:
import ipywidgets as wdg
import utilities
from common import WOOD_MS_DEFAULTS_FILTER
from wood_microstructure.cli.postproc import filter_porosity

widget_filter, _ = utilities.click_cli_widget(
    filter_porosity,
    override_defaults=WOOD_MS_DEFAULTS_FILTER,
    visible_panel_names=['Options'],
    title='Filter porosity',
    button_description='Filter',
    button_running_text='Filtering...'
)
filter_box = wdg.VBox([widget_filter])
# filter_accordion = wdg.Accordion(children=[widget_filter], selected_index=None)
# filter_accordion.set_title(0, "Filter Porosity")
# display(filter_accordion)

In [ ]:
import re
import os

import numpy as np
from IPython.display import Image as IPyImage
import matplotlib.pyplot as plt

rgx = re.compile(r"loss_curve_fold_(\d+).dat")

def loss_finder(base_path: str) -> list[tuple[str, list[str]]]:
    """Find all loss functions and group them by directory (one directory can include losses for multiple folds in the
    same training run).
    The .dat files have 3 columns: epoch, training loss, validation loss.
    """
    found_losses: dict[str, list[str]] = {}
    for root, dirs, files in os.walk(base_path):
        for file in files:
            match = rgx.match(file)
            if match:
                loss_dir = os.path.basename(root)
                loss_path = os.path.join(root, file)
                ptr = found_losses.setdefault(loss_dir, [])
                ptr.append(loss_path)

    valid_dirs = []
    for loss_dir, loss_files in found_losses.items():
        _, res = loss_loader(loss_files)
        if res is not None and res.size > 0:
            valid_dirs.append((loss_dir, loss_files))
            
    return valid_dirs

def loss_loader(loss_files: list[str]) -> tuple[str, np.ndarray]:
    """Load the data by keeping the epoch as the first column and the train/valid loss ordered per fold."""
    sorted_files = sorted(loss_files, key=lambda x: int(rgx.match(os.path.basename(x)).group(1)))
    res = None
    for file in sorted_files:
        basedir = os.path.dirname(file)
        file_data = np.loadtxt(file)
        if file_data.ndim != 2 or file_data.shape[1] < 3:
            # print(f"Warning: file {file} has unexpected shape {file_data.shape}. Expected 2D array with at least 3 columns.")
            continue
        # print(f"Loaded loss data from {file}: shape={file_data.shape}")
        if res is None:
            res = file_data
        else:
            if res.shape[0] != file_data.shape[0] or not np.allclose(res[:, 0], file_data[:, 0]):
                raise ValueError(f"Epochs do not match between files: {res.shape[0]} vs {file_data.shape[0]}")
            res = np.hstack((res, file_data[:, 1:]))

    return basedir, res

def loss_visualizer(loss_dir_data: tuple[str, np.ndarray]): 
    """Visualize the loss function with mean and standard deviation across folds."""
    loss_dir, loss_data = loss_dir_data
    if loss_data is None or loss_data.size == 0:
        # print(f"No loss data to visualize for {loss_dir}.")
        return
    title = f"Loss function for {loss_dir}"
    xlabel = "Epochs"
    ylabel = "Loss"
    epochs = loss_data[:, 0]
    train_losses = loss_data[:, 1::2]
    valid_losses = loss_data[:, 2::2]

    train_mean = np.mean(train_losses, axis=1)
    train_std = np.std(train_losses, axis=1)
    valid_mean = np.mean(valid_losses, axis=1)
    valid_std = np.std(valid_losses, axis=1)

    plt.figure(figsize=(6, 3.5))
    plt.plot(epochs, train_mean, label='Training Loss', color='blue')
    plt.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.2)
    plt.plot(epochs, valid_mean, label='Validation Loss', color='orange')
    plt.fill_between(epochs, valid_mean - valid_std, valid_mean + valid_std, color='orange', alpha=0.2)

    plt.yscale('log')

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid()
    plt.tight_layout()

    filename = os.path.join(loss_dir, "loss_curve.png")
    plt.savefig(filename, dpi=200)

    image = IPyImage(filename=filename)
    display(image)
    print(f"Saved loss curve to {filename}")

In [ ]:
import ipywidgets as wdg
import utilities
from common import WOOD_MS_DEFAULTS_TRAIN, WOOD_MS_OUTDIR
from wood_microstructure.cli.train import train

widget_train, train_event = utilities.click_cli_widget(
    train,
    title='Train surrogate model',
    override_defaults=WOOD_MS_DEFAULTS_TRAIN,
    visible_panel_names=['Options'],
    custom_groupings={
        None: ['train_dir']
    },
    button_description='Train',
    button_running_text='Training...',
)
train_visualizer_accordion = utilities.output_visualizer_widget(
    title='Visualize training outputs',
    event_observer=train_event,
    # output_dir=WOOD_MS_OUTDIR,
    outputs_func=lambda: loss_finder(WOOD_MS_OUTDIR),
    read_func=loss_loader,
    data_display_func=loss_visualizer,
)
train_vbox = wdg.VBox([widget_train, train_visualizer_accordion])

# train_accordion = wdg.Accordion(children=[train_vbox], selected_index=None)
# train_accordion.set_title(0, "Train Surrogate Model")
# display(train_accordion)

In [ ]:
main_accordion = wdg.Accordion(children=[generate_box, filter_box, train_vbox], selected_index=0)
main_accordion.set_title(0, "Generate Microstructure")
main_accordion.set_title(1, "Filter Porosity")
main_accordion.set_title(2, "Train Surrogate Model")
display(main_accordion)

In [ ]:
import utilities
utilities.final_spacer()